In [22]:
import pandas as pd
from pyprojroot import here

In [23]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [24]:
crime = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')
population = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv")

In [25]:
crime['cvemun'] = crime['Cve. Municipio'] - crime['Clave_Ent'] * 1000
rows_to_drop = crime[crime['cvemun'] > 600].index
crime = crime.drop(index=rows_to_drop)

In [26]:
crime['CVEGEO'] = crime['Cve. Municipio'].astype(str).str.zfill(5)
population['CVEGEO'] = population['CVEGEO'].astype(str).str.zfill(5)

In [27]:
crime_2025 = crime[crime['Año'] == 2025].copy()
total_muns_2025 = crime_2025['CVEGEO'].unique().tolist()
len(total_muns_2025)

2478

In [28]:
crime = crime.rename(columns={'Año': 'year'})
crime['total'] = crime.iloc[:, 9:21].sum(axis=1)

In [29]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

B_crime = [ # Criminal activity associated with organized crime by its general activity
    'Homicidio doloso',
    'Extorsión',
    'Evasión de presos'
]

C_crime = [ # Logistics related crime
    'Robo a transportista',
    'Robo de vehículo automotor'
]

D_crime = [
    'Delitos cometidos por servidores públicos',
    'Electorales'
]

In [30]:
A_df = crime[crime['Subtipo de delito'].isin(A_crime)].copy()
AB_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime)].copy()
ABC_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime)].copy()
ABCD_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime + D_crime)].copy()

In [31]:
grouped_AB = AB_df.groupby(['year', 'CVEGEO'])['total'].sum().drop(columns='Unnamed: 0').reset_index()

In [32]:
df = grouped_AB

munis_2025 = df[df['year'] == 2025][['CVEGEO']].drop_duplicates()

years = df['year'].unique()
skeleton = munis_2025.assign(key=1).merge(
    pd.DataFrame({'year': years, 'key': 1}), on='key'
).drop(columns='key')

df_fixed = skeleton.merge(df, on=['CVEGEO', 'year'], how='left')
df_fixed['total'] = df_fixed['total'].fillna(0).astype(int)
df_fixed = df_fixed.sort_values(['year', 'CVEGEO']).reset_index(drop=True)

In [33]:
result_AB = df_fixed.merge(population, on=['year', 'CVEGEO'], how='left')
result_AB.drop(columns='Unnamed: 0', inplace=True)
result_AB['crime_rate'] = result_AB['total'] / result_AB['population'] * 100000

In [34]:
result_AB.to_csv(INTERIM_DATA_DIR/"crime_rates_AB.csv", encoding='latin-1')

print(f" {len(result_AB)} municipalities processed")
print(result_AB.sample(10))

 27258 municipalities processed
      CVEGEO  year  total  population  crime_rate
18115  15100  2022     26     98824.8   26.309186
21501  21093  2023      1     15249.3    6.557678
19732  31073  2022      0     12298.4    0.000000
7840   12031  2018      0     12637.0    0.000000
15249  12006  2021      0     22041.4    0.000000
15288  12045  2021      2     57858.2    3.456727
1203   20189  2015      0      5633.0    0.000000
9988   06003  2019     27     43428.5   62.171155
3491   19050  2016      3      3613.8   83.015109
928    17021  2015      9     15081.0   59.677740
